In [3]:
"""
Comparative Analysis of Discriminative vs. Generative Algorithms
Dataset: Wisconsin Breast Cancer Diagnostic Dataset

This script evaluates Logistic Regression, Gaussian Discriminant Analysis (GDA),
and Gaussian Naive Bayes to analyze their performance and extract their
learned mathematical parameters.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, Any

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix

DATA_FILEPATH = 'data.csv'
TARGET_COLUMN = 'diagnosis'
RANDOM_SEED = 42
TEST_SPLIT_SIZE = 0.2
MAX_ITERATIONS = 1000
OUTPUT_DIR = 'experiment_results'

def setup_environment() -> None:
    """Creates necessary directories for saving artifacts like plots."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_and_preprocess_data(filepath: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list]:
    """
    Loads dataset, handles missing values, encodes target, and standardizes features.
    """
    print("--- Data Loading & Preprocessing ---")
    df = pd.read_csv(filepath)

    # Drop non-predictive features (ID and empty columns)
    df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')

    X = df.drop(columns=[TARGET_COLUMN])
    feature_names = X.columns.tolist()

    # Encode target: Benign (B) -> 0, Malignant (M) -> 1
    y = LabelEncoder().fit_transform(df[TARGET_COLUMN])

    # Stratified split to maintain class distributions
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SPLIT_SIZE, random_state=RANDOM_SEED, stratify=y
    )

    # Standardize features (Mean=0, Variance=1)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Training samples: {X_train_scaled.shape[0]}, Features: {X_train_scaled.shape[1]}")
    print(f"Testing samples: {X_test_scaled.shape[0]}\n")

    return X_train_scaled, X_test_scaled, y_train, y_test, feature_names

def extract_model_parameters(name: str, model: Any, feature_names: list) -> None:
    """
    Extracts and prints the underlying mathematical parameters learned by the models.
    """
    print(f"\n--- Learned Parameters: {name} ---")

    if name == "Logistic Regression":
        # Extracts theta_0 (intercept) and theta_1...n (feature weights)
        print(f"Intercept (\u03B8\u2080): {model.intercept_[0]:.4f}")
        print("Top 3 Feature Weights:")
        weights = model.coef_[0]
        top_indices = np.argsort(np.abs(weights))[-3:][::-1]
        for idx in top_indices:
            print(f"  {feature_names[idx]}: {weights[idx]:.4f}")

    elif name == "GDA (LDA)":
        # Extracts the class priors (phi) and the means (mu)
        print(f"Class Priors (\u03D5): Benign={model.priors_[0]:.4f}, Malignant={model.priors_[1]:.4f}")
        print("Centroid Means (\u03BC) for top feature (radius_mean):")
        print(f"  Benign: {model.means_[0][0]:.4f}")
        print(f"  Malignant: {model.means_[1][0]:.4f}")

    elif name == "Naive Bayes":
        # Extracts the independent variances (sigma^2) and means (mu)
        print(f"Class Priors: Benign={model.class_prior_[0]:.4f}, Malignant={model.class_prior_[1]:.4f}")
        print("Variance (\u03C3\u00B2) for top feature (radius_mean):")
        print(f"  Benign: {model.var_[0][0]:.4f}")
        print(f"  Malignant: {model.var_[1][0]:.4f}")

def evaluate_and_plot(models_dict: Dict[str, Any], X_test: np.ndarray, y_test: np.ndarray) -> None:
    """
    Evaluates models, prints metrics, and generates a comparative confusion matrix diagram.
    """
    print("\n--- Model Evaluation ---")

    # Set plotting style
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Confusion Matrices: Model Comparison", fontsize=16, fontweight='bold', y=1.05)

    for ax, (name, model) in zip(axes, models_dict.items()):
        y_pred = model.predict(X_test)

        # Calculate Metrics
        acc = accuracy_score(y_test, y_pred)
        print(f"{name} Accuracy: {acc * 100:.2f}%")

        # Generate Confusion Matrix Plot
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                    annot_kws={"size": 14, "weight": "bold"},
                    xticklabels=['Benign (0)', 'Malignant (1)'],
                    yticklabels=['Benign (0)', 'Malignant (1)'])
        ax.set_title(f"{name}\nAccuracy: {acc*100:.2f}%", fontsize=14, pad=10)
        ax.set_xlabel("Predicted Label", fontsize=12)
        ax.set_ylabel("True Label", fontsize=12)

    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, 'confusion_matrices.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"\nDiagnostic diagrams successfully saved to: {plot_path}")

def main():
    setup_environment()

    try:
        X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data(DATA_FILEPATH)
    except FileNotFoundError:
        print(f"\n[ERROR] Could not find '{DATA_FILEPATH}'.")
        print("Please ensure your dataset is named 'data.csv' and is located in the exact same folder as this script.")
        return

    # Define models with explicit hyperparameters
    models = {
        "Logistic Regression": LogisticRegression(max_iter=MAX_ITERATIONS, random_state=RANDOM_SEED),
        "GDA (LDA)": LinearDiscriminantAnalysis(),
        "Naive Bayes": GaussianNB()
    }

    # Train models and extract their mathematical parameters
    for name, model in models.items():
        model.fit(X_train, y_train)
        extract_model_parameters(name, model, feature_names)

    # Evaluate accuracy and generate visual confusion matrices
    evaluate_and_plot(models, X_test, y_test)

if __name__ == "__main__":
    main()

--- Data Loading & Preprocessing ---
Training samples: 455, Features: 30
Testing samples: 114


--- Learned Parameters: Logistic Regression ---
Intercept (θ₀): -0.2430
Top 3 Feature Weights:
  texture_worst: 1.4341
  radius_se: 1.2333
  symmetry_worst: 1.0613

--- Learned Parameters: GDA (LDA) ---
Class Priors (ϕ): Benign=0.6264, Malignant=0.3736
Centroid Means (μ) for top feature (radius_mean):
  Benign: -0.5624
  Malignant: 0.9428

--- Learned Parameters: Naive Bayes ---
Class Priors: Benign=0.6264, Malignant=0.3736
Variance (σ²) for top feature (radius_mean):
  Benign: 0.2477
  Malignant: 0.8422

--- Model Evaluation ---
Logistic Regression Accuracy: 96.49%
GDA (LDA) Accuracy: 96.49%
Naive Bayes Accuracy: 92.11%

Diagnostic diagrams successfully saved to: experiment_results/confusion_matrices.png
